In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
path = "C:\\Users\\brand\\OneDrive\\Documents\\Time Series Project\\Anomaly-Detection-Thesis\\Benchmark CSVs\\"

MOE_df = pd.read_csv(path + "univariate_moe.csv")
transformer_df = pd.read_csv(path + "univariate_transformer.csv")

In [ ]:
columns = ['Time', 'AUC-PR', 'AUC-ROC', 'VUS-PR', 'VUS-ROC', 'Standard-F1', 'PA-F1', 'Event-based-F1', 'R-based-F1', 'Affiliation-F']
all_models = [transformer_df, MOE_df]

# 3. Create a matching list of names for labeling
model_names = ['Transformer', 'Moe']

summary_list = []

for df, name in zip(all_models, model_names):
    # Calculate the mean of the metrics for this model
    means = df[columns].mean()
    means.name = name  # Set the name of the row to the model name
    summary_list.append(means)

# Combine into one dataframe
comparison_table = pd.concat(summary_list, axis=1).T

print("\n\n--- Transformer vs. MOE TABLE (AVERAGES) ---")
# Adjust pandas settings to ensure the whole table prints
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(comparison_table.round(4))

In [ ]:
# 1. Extract the source name into a permanent column
transformer_df['source'] = transformer_df['file'].str.extract(r'\d+_([a-zA-Z]+)')

# 2. Split into a dictionary: keys are 'Genesis', 'NASA', etc.
# values are the corresponding dataframes
transformer_groups = {name: group for name, group in transformer_df.groupby('source')}


# 1. Extract the source name into a permanent column
MOE_df['source'] = MOE_df['file'].str.extract(r'\d+_([a-zA-Z]+)')

# 2. Split into a dictionary: keys are 'Genesis', 'NASA', etc.
# values are the corresponding dataframes
MOE_groups = {name: group for name, group in MOE_df.groupby('source')}

MOE_df['source'] = MOE_df['file'].str.extract(r'\d+_([a-zA-Z]+)')

# 2. Split into a dictionary: keys are 'Genesis', 'NASA', etc.
# values are the corresponding dataframes
MOE2_groups2 = {name: group for name, group in MOE_df.groupby('source')}

In [ ]:
# Define your metrics
columns = ['Time', 'AUC-PR', 'AUC-ROC', 'VUS-PR', 'VUS-ROC', 'Standard-F1', 'PA-F1', 'Event-based-F1', 'R-based-F1', 'Affiliation-F']
datasets = ['Genesis', 'MSL', 'Daphnet', 'MITDB', 'GHL', 'SMD', 'LTDB', 'SVDB', 'PSM', 'TAO', 'OPPORTUNITY', 'CreditCard', 'CATSv2', 'SMAP', 'GECCO', 'Exathlon']

summary_data = []

for ds in datasets:
    # Safely get the dataframes from your dictionaries
    # (Assumes transformer_groups and MOE_groups are dicts keyed by dataset name)
    df_trans = transformer_groups.get(ds)
    df_moe = MOE_groups.get(ds)
    
    # Skip if the dataset is missing in either model's results
    if df_trans is None or df_moe is None:
        continue

    # 1. Calculate Averages for both for this dataset
    mean_trans = df_trans[columns].mean()
    mean_moe = df_moe[columns].mean()

    # 2. Calculate Deltas (MoE - Transformer)
    # Positive means MoE performed better
    auc_roc_delta = mean_moe['AUC-ROC'] - mean_trans['AUC-ROC']
    vus_pr_delta = mean_moe['VUS-PR'] - mean_trans['VUS-PR']
    Event_F1_delta = mean_moe['Event-based-F1'] - mean_trans['Event-based-F1']
    VUS_ROC_Delta = mean_moe['VUS-ROC'] - mean_trans['VUS-ROC']
    
    # 3. Efficiency Loss (Relative increase in time)
    # (MoE_Time / Trans_Time) - 1. E.g., 0.36 means MoE is 36% slower.
    eff_loss = ((mean_trans['Time'] / mean_moe['Time']) - 1) * 100

    # Store results in a flat dictionary
    summary_data.append({
        'Dataset': ds,
        'Trans_AUC-ROC': mean_trans['AUC-ROC'],
        'MoE_AUC-ROC': mean_moe['AUC-ROC'],
        'Trans_event_F1': mean_trans['Event-based-F1'],
        'MoE_event_F1': mean_moe['Event-based-F1'],
        'AUC-ROC_Delta': auc_roc_delta,
        'VUS-PR_Delta': vus_pr_delta,
        'VUS-ROC_Delta': VUS_ROC_Delta,
        'Event_F1_Delta': Event_F1_delta,
        'Efficiency_Loss': eff_loss,
        'Trans_Time': mean_trans['Time'],
        'MoE_Time': mean_moe['Time']
    })

# Create the final dataframe
comparison_table = pd.DataFrame(summary_data).set_index('Dataset')

print("\n--- Granular Performance Comparison ---")
with pd.option_context('display.max_columns', None, 'display.width', 1000):
    print(comparison_table.round(4))